# Transformer Encoders Demo

This notebook demonstrates the Tabular Transformer Encoder and Vision Transformer Encoder used in the Fraud Detection System.

## Overview

We will explore:
1. **Tabular Transformer Encoder**: Processes transaction features using self-attention
2. **Vision Transformer Encoder**: Processes QR code images using patch-based encoding

Each encoder will be demonstrated with sample inputs and outputs.

## 1. Setup and Imports

In [ ]:
# Install dependencies (uncomment if running in Google Colab)
# !pip install tensorflow numpy matplotlib

import sys
import os

# If running in Colab, clone the repository
# !git clone https://github.com/go2nishantnig/test.git
# sys.path.append('/content/test')

# If running locally, add parent directory to path
sys.path.append('..')

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

## 2. Import Model Components

In [ ]:
from src.models.blocks import TransformerBlock
from src.models.embeddings import PatchEmbedding
from src.models.attention import MultiHeadSelfAttention

print("Model components imported successfully!")

## 3. Tabular Transformer Encoder

The Tabular Transformer Encoder processes transaction features (like amount, distance, merchant info) through self-attention layers.

### Architecture:
1. **Input Projection**: Projects features to model dimension (d_model)
2. **Positional Encoding**: Adds position information
3. **Transformer Blocks**: Multi-head self-attention + Feed-forward network
4. **Output**: Contextualized feature representations

In [ ]:
# Configuration for Tabular Encoder
tabular_config = {
    'num_features': 8,           # Number of input features (e.g., amount, distance, etc.)
    'max_sequence_length': 1,    # Sequence length (1 for single transaction)
    'd_model': 128,              # Model dimension
    'num_heads': 8,              # Number of attention heads
    'num_layers': 4,             # Number of transformer blocks
    'dff': 512,                  # Feed-forward network dimension
    'dropout_rate': 0.1          # Dropout rate
}

print("Tabular Encoder Configuration:")
for key, value in tabular_config.items():
    print(f"  {key}: {value}")

In [ ]:
# Build Tabular Transformer Encoder
def build_tabular_encoder(config):
    """
    Build a tabular transformer encoder
    
    Args:
        config: Configuration dictionary
    
    Returns:
        Keras model for tabular encoding
    """
    # Input layer for tabular features
    inputs = layers.Input(
        shape=(config['max_sequence_length'], config['num_features']),
        name='tabular_input'
    )
    
    # Project input to d_model dimension
    x = layers.Dense(config['d_model'], name='feature_projection')(inputs)
    
    # Add positional encoding
    position_ids = tf.constant([list(range(config['max_sequence_length']))])
    position_embedding_layer = layers.Embedding(
        input_dim=config['max_sequence_length'],
        output_dim=config['d_model'],
        name='position_embedding'
    )
    position_embeddings = position_embedding_layer(position_ids)
    x = x + position_embeddings
    
    # Stack transformer blocks
    for i in range(config['num_layers']):
        x = TransformerBlock(
            d_model=config['d_model'],
            num_heads=config['num_heads'],
            dff=config['dff'],
            dropout_rate=config['dropout_rate'],
            name=f'transformer_block_{i}'
        )(x)
    
    # Create model
    model = keras.Model(inputs=inputs, outputs=x, name='tabular_transformer_encoder')
    return model

# Build the tabular encoder
tabular_encoder = build_tabular_encoder(tabular_config)
print("\nTabular Transformer Encoder built successfully!")
print(f"\nModel Summary:")
tabular_encoder.summary()

In [ ]:
# Create sample tabular data (transaction features)
batch_size = 4
sample_tabular_data = np.random.randn(
    batch_size,
    tabular_config['max_sequence_length'],
    tabular_config['num_features']
).astype(np.float32)

print("Input Shape:", sample_tabular_data.shape)
print("\nSample Input Data (first transaction):")
print(sample_tabular_data[0])

# Pass through encoder
tabular_output = tabular_encoder(sample_tabular_data, training=False)

print("\n" + "="*60)
print("TABULAR TRANSFORMER ENCODER OUTPUT")
print("="*60)
print(f"Output Shape: {tabular_output.shape}")
print(f"Output (first transaction encoded features):")
print(tabular_output[0].numpy())
print(f"\nOutput Statistics:")
print(f"  Mean: {tf.reduce_mean(tabular_output).numpy():.4f}")
print(f"  Std:  {tf.math.reduce_std(tabular_output).numpy():.4f}")
print(f"  Min:  {tf.reduce_min(tabular_output).numpy():.4f}")
print(f"  Max:  {tf.reduce_max(tabular_output).numpy():.4f}")

In [ ]:
# Visualize tabular encoder output
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Plot 1: Heatmap of encoded features for all samples
output_flat = tf.reshape(tabular_output, [batch_size, -1]).numpy()
im = axes[0].imshow(output_flat, cmap='viridis', aspect='auto')
axes[0].set_title('Tabular Encoder Output (All Samples)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Feature Dimension')
axes[0].set_ylabel('Sample Index')
plt.colorbar(im, ax=axes[0])

# Plot 2: Distribution of output values
axes[1].hist(output_flat.flatten(), bins=50, color='skyblue', edgecolor='black', alpha=0.7)
axes[1].set_title('Distribution of Encoded Feature Values', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Tabular Transformer Encoder successfully processes transaction features!")

## 4. Vision Transformer Encoder

The Vision Transformer (ViT) Encoder processes QR code images by:
1. Dividing images into patches
2. Flattening and embedding each patch
3. Adding positional embeddings
4. Processing through transformer blocks

### Architecture:
1. **Patch Extraction**: Divide image into non-overlapping patches
2. **Patch Embedding**: Flatten and project patches to d_model dimension
3. **Positional Encoding**: Add learnable position embeddings
4. **Transformer Blocks**: Multi-head self-attention + Feed-forward network
5. **Output**: Contextualized patch representations

In [ ]:
# Configuration for Vision Transformer Encoder
vision_config = {
    'image_size': (128, 128),    # Image dimensions (height, width)
    'image_channels': 3,         # Number of channels (RGB)
    'patch_size': 16,            # Size of each patch (16x16)
    'd_model': 128,              # Model dimension
    'num_heads': 8,              # Number of attention heads
    'num_layers': 4,             # Number of transformer blocks
    'dff': 512,                  # Feed-forward network dimension
    'dropout_rate': 0.1          # Dropout rate
}

# Calculate number of patches
num_patches = (vision_config['image_size'][0] // vision_config['patch_size']) * \
              (vision_config['image_size'][1] // vision_config['patch_size'])
vision_config['num_patches'] = num_patches

print("Vision Transformer Encoder Configuration:")
for key, value in vision_config.items():
    print(f"  {key}: {value}")
print(f"\nNote: {vision_config['image_size'][0]}x{vision_config['image_size'][1]} image with {vision_config['patch_size']}x{vision_config['patch_size']} patches = {num_patches} patches")

In [ ]:
# Build Vision Transformer Encoder
def build_vision_encoder(config):
    """
    Build a vision transformer encoder
    
    Args:
        config: Configuration dictionary
    
    Returns:
        Keras model for image encoding
    """
    # Input layer for images
    image_inputs = layers.Input(
        shape=(*config['image_size'], config['image_channels']),
        name='image_input'
    )
    
    # Extract and embed patches
    patch_embedding = PatchEmbedding(
        image_size=config['image_size'],
        patch_size=config['patch_size'],
        d_model=config['d_model'],
        name='patch_embedding'
    )
    x = patch_embedding(image_inputs)
    
    # Stack transformer blocks for image patches
    for i in range(config['num_layers']):
        x = TransformerBlock(
            d_model=config['d_model'],
            num_heads=config['num_heads'],
            dff=config['dff'],
            dropout_rate=config['dropout_rate'],
            name=f'vit_transformer_block_{i}'
        )(x)
    
    # Create model
    model = keras.Model(inputs=image_inputs, outputs=x, name='vision_transformer_encoder')
    return model

# Build the vision encoder
vision_encoder = build_vision_encoder(vision_config)
print("\nVision Transformer Encoder built successfully!")
print(f"\nModel Summary:")
vision_encoder.summary()

In [ ]:
# Create sample image data (QR codes)
batch_size = 4
sample_images = np.random.rand(
    batch_size,
    vision_config['image_size'][0],
    vision_config['image_size'][1],
    vision_config['image_channels']
).astype(np.float32)

print("Input Shape:", sample_images.shape)
print(f"Input: {batch_size} images of size {vision_config['image_size']} with {vision_config['image_channels']} channels")

# Pass through encoder
vision_output = vision_encoder(sample_images, training=False)

print("\n" + "="*60)
print("VISION TRANSFORMER ENCODER OUTPUT")
print("="*60)
print(f"Output Shape: {vision_output.shape}")
print(f"  - Batch Size: {vision_output.shape[0]}")
print(f"  - Number of Patches: {vision_output.shape[1]}")
print(f"  - Embedding Dimension: {vision_output.shape[2]}")
print(f"\nOutput (first image, first 5 patches):")
print(vision_output[0, :5, :].numpy())
print(f"\nOutput Statistics:")
print(f"  Mean: {tf.reduce_mean(vision_output).numpy():.4f}")
print(f"  Std:  {tf.math.reduce_std(vision_output).numpy():.4f}")
print(f"  Min:  {tf.reduce_min(vision_output).numpy():.4f}")
print(f"  Max:  {tf.reduce_max(vision_output).numpy():.4f}")

In [ ]:
# Visualize vision encoder
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Plot 1: Sample input image
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(sample_images[0])
ax1.set_title('Sample Input Image (QR Code)', fontsize=12, fontweight='bold')
ax1.axis('off')

# Plot 2: Grid showing patch boundaries
ax2 = fig.add_subplot(gs[0, 1])
img_with_grid = sample_images[0].copy()
ax2.imshow(img_with_grid)
patch_size = vision_config['patch_size']
for i in range(0, vision_config['image_size'][0], patch_size):
    ax2.axhline(y=i, color='red', linewidth=1, alpha=0.7)
for j in range(0, vision_config['image_size'][1], patch_size):
    ax2.axvline(x=j, color='red', linewidth=1, alpha=0.7)
ax2.set_title(f'Image Divided into {num_patches} Patches', fontsize=12, fontweight='bold')
ax2.axis('off')

# Plot 3: Number of patches
ax3 = fig.add_subplot(gs[0, 2])
ax3.text(0.5, 0.5, f'{num_patches}\nPatches', 
         ha='center', va='center', fontsize=36, fontweight='bold', color='darkblue')
ax3.set_title('Total Patches per Image', fontsize=12, fontweight='bold')
ax3.axis('off')

# Plot 4: Heatmap of encoded patches (first image)
ax4 = fig.add_subplot(gs[1, :])
output_first_image = vision_output[0].numpy()  # Shape: (num_patches, d_model)
im = ax4.imshow(output_first_image.T, cmap='viridis', aspect='auto')
ax4.set_title('Vision Encoder Output - Encoded Patch Features (First Image)', 
              fontsize=12, fontweight='bold')
ax4.set_xlabel('Patch Index')
ax4.set_ylabel('Feature Dimension')
plt.colorbar(im, ax=ax4, label='Activation Value')

plt.tight_layout()
plt.show()

print("\n✓ Vision Transformer Encoder successfully processes QR code images!")

## 5. Encoder Comparison

Let's compare the two encoders side by side:

In [ ]:
print("="*70)
print("ENCODER COMPARISON")
print("="*70)
print()
print(f"{'Aspect':<30} {'Tabular Encoder':<20} {'Vision Encoder':<20}")
print("-"*70)
print(f"{'Input Type':<30} {'Transaction Features':<20} {'QR Code Images':<20}")
print(f"{'Input Shape':<30} {str(sample_tabular_data.shape):<20} {str(sample_images.shape):<20}")
print(f"{'Output Shape':<30} {str(tuple(tabular_output.shape)):<20} {str(tuple(vision_output.shape)):<20}")
print(f"{'Sequence Length':<30} {tabular_config['max_sequence_length']:<20} {num_patches:<20}")
print(f"{'Model Dimension (d_model)':<30} {tabular_config['d_model']:<20} {vision_config['d_model']:<20}")
print(f"{'Number of Layers':<30} {tabular_config['num_layers']:<20} {vision_config['num_layers']:<20}")
print(f"{'Number of Heads':<30} {tabular_config['num_heads']:<20} {vision_config['num_heads']:<20}")
print(f"{'Parameters':<30} {tabular_encoder.count_params():<20} {vision_encoder.count_params():<20}")
print()
print("Key Differences:")
print("  • Tabular Encoder: Processes structured numerical features")
print("  • Vision Encoder: Uses patch-based approach for image understanding")
print("  • Both use the same transformer architecture internally")
print("  • Vision Encoder has more tokens (patches) to process")
print("="*70)

## 6. Combined Visualization

In [ ]:
# Create a comprehensive comparison plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Top Left: Tabular Output Heatmap
tabular_flat = tf.reshape(tabular_output, [batch_size, -1]).numpy()
im1 = axes[0, 0].imshow(tabular_flat, cmap='coolwarm', aspect='auto')
axes[0, 0].set_title('Tabular Encoder Output\n(Transaction Features)', 
                     fontsize=14, fontweight='bold')
axes[0, 0].set_xlabel('Feature Dimension')
axes[0, 0].set_ylabel('Sample Index')
plt.colorbar(im1, ax=axes[0, 0])

# Top Right: Vision Output Heatmap (first image)
vision_first = vision_output[0].numpy()
im2 = axes[0, 1].imshow(vision_first.T, cmap='coolwarm', aspect='auto')
axes[0, 1].set_title('Vision Encoder Output\n(QR Code Patches)', 
                     fontsize=14, fontweight='bold')
axes[0, 1].set_xlabel('Patch Index')
axes[0, 1].set_ylabel('Feature Dimension')
plt.colorbar(im2, ax=axes[0, 1])

# Bottom Left: Distribution comparison
axes[1, 0].hist(tabular_flat.flatten(), bins=50, alpha=0.7, label='Tabular', color='blue')
axes[1, 0].hist(vision_output[0].numpy().flatten(), bins=50, alpha=0.7, label='Vision', color='green')
axes[1, 0].set_title('Distribution of Encoded Values', fontsize=14, fontweight='bold')
axes[1, 0].set_xlabel('Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Bottom Right: Statistics comparison
tabular_stats = [
    tf.reduce_mean(tabular_output).numpy(),
    tf.math.reduce_std(tabular_output).numpy(),
    tf.reduce_min(tabular_output).numpy(),
    tf.reduce_max(tabular_output).numpy()
]
vision_stats = [
    tf.reduce_mean(vision_output).numpy(),
    tf.math.reduce_std(vision_output).numpy(),
    tf.reduce_min(vision_output).numpy(),
    tf.reduce_max(vision_output).numpy()
]

x_pos = np.arange(4)
width = 0.35

axes[1, 1].bar(x_pos - width/2, tabular_stats, width, label='Tabular', color='blue', alpha=0.7)
axes[1, 1].bar(x_pos + width/2, vision_stats, width, label='Vision', color='green', alpha=0.7)
axes[1, 1].set_title('Statistics Comparison', fontsize=14, fontweight='bold')
axes[1, 1].set_xticks(x_pos)
axes[1, 1].set_xticklabels(['Mean', 'Std', 'Min', 'Max'])
axes[1, 1].set_ylabel('Value')
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("DEMONSTRATION COMPLETE!")
print("="*70)
print("Both encoders have successfully processed their respective inputs:")
print("  ✓ Tabular Transformer Encoder: Transaction features → Encoded representations")
print("  ✓ Vision Transformer Encoder: QR code images → Encoded patch features")
print("\nThese encoders form the foundation of the Multimodal Fraud Detection System.")
print("="*70)

## 7. Conclusion

This notebook demonstrated:

### Tabular Transformer Encoder
- Processes structured transaction features (amount, distance, merchant info)
- Uses self-attention to capture relationships between features
- Output: Contextualized feature representations of shape `(batch_size, sequence_length, d_model)`

### Vision Transformer Encoder
- Processes QR code images using a patch-based approach
- Divides images into patches and treats them as tokens
- Uses self-attention to understand spatial relationships
- Output: Encoded patch features of shape `(batch_size, num_patches, d_model)`

### Key Takeaways
1. Both encoders use the same transformer architecture
2. The main difference is in input preprocessing:
   - Tabular: Direct feature projection
   - Vision: Patch extraction and embedding
3. Both produce rich, contextualized representations
4. These representations can be fused for multimodal fraud detection

### Next Steps
- Explore cross-modal attention fusion between the two encoders
- Train the complete multimodal model on real data
- Analyze attention patterns to understand model decisions

---

**For more information, see the full repository:** [Multimodal Fraud Detection Transformer](https://github.com/go2nishantnig/test)